# 🏥 Survival Analysis in Healthcare
**Author:** Felipe Taha Sant'Ana
**Dataset:** 4,000 patients, 14 features, event + censoring data

---
## Overview
Time-to-event analysis for clinical outcome prediction. Includes Kaplan-Meier survival curves stratified by treatment/stage/smoking, Cox-like hazard ratio estimation, risk stratification into Low/Medium/High groups, and calibration analysis.

## Contents
1. Patient Data — 2. Kaplan-Meier Analysis — 3. Hazard Ratios — 4. Predictive Modeling — 5. Risk Stratification — 6. Conclusions

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid"); COLORS = {'primary':'#059669','secondary':'#6366F1','accent':'#F59E0B','danger':'#EF4444','info':'#0EA5E9','dark':'#0F172A'}
palette = list(COLORS.values())
np.random.seed(42)

# ── Generate patient survival data ──
n = 4000
age = np.random.normal(62,12,n).clip(30,90).astype(int)
gender = np.random.choice(['Male','Female'],n,p=[0.55,0.45])
treatment = np.random.choice(['Treatment_A','Treatment_B','Control'],n,p=[0.35,0.35,0.30])
stage = np.random.choice(['I','II','III','IV'],n,p=[0.20,0.30,0.30,0.20])
bmi = np.random.normal(27,5,n).clip(16,45).round(1)
smoking = np.random.choice(['Never','Former','Current'],n,p=[0.40,0.35,0.25])
comorbidities = np.random.poisson(1.5,n).clip(0,6)
biomarker_a = np.random.lognormal(2.0,0.8,n).round(1)
biomarker_b = np.random.normal(50,15,n).clip(10,100).round(1)
perf_score = np.random.choice([0,1,2,3],n,p=[0.25,0.35,0.25,0.15])

log_h = (-3.5+0.03*(age-60)+0.15*(gender=='Male')-0.4*(treatment=='Treatment_A')-0.2*(treatment=='Treatment_B')
    +0.3*(stage=='II')+0.7*(stage=='III')+1.2*(stage=='IV')+0.02*(bmi-25)
    +0.2*(smoking=='Former')+0.5*(smoking=='Current')+0.15*comorbidities+0.001*biomarker_a+0.01*(biomarker_b-50)+0.25*perf_score)
hazard = np.exp(log_h)
surv_time = np.clip(np.random.weibull(1.2,n)*(1.0/hazard)*30, 1, 2000)
cens_time = np.random.uniform(100, 1825, n)
obs_time = np.minimum(surv_time, cens_time)
event = (surv_time <= cens_time).astype(int)

df = pd.DataFrame({'PatientID':[f'PT-{i:05d}' for i in range(n)],'Age':age,'Gender':gender,'Treatment':treatment,
    'Stage':stage,'BMI':bmi,'Smoking':smoking,'Comorbidities':comorbidities,'BiomarkerA':biomarker_a,
    'BiomarkerB':biomarker_b,'PerformanceScore':perf_score,'Time_Days':np.round(obs_time).astype(int),
    'Time_Months':np.round(obs_time/30.44,1),'Event':event})
print(f"Patients: {len(df):,} | Event rate: {df['Event'].mean():.1%}")
df.head()

In [ ]:
def kaplan_meier(time, event):
    df_km = pd.DataFrame({'time':time,'event':event}).sort_values('time')
    unique_t = np.sort(df_km['time'].unique())
    surv, n = 1.0, len(df_km)
    t_out, s_out = [0], [1.0]
    for t in unique_t:
        at_t = df_km[df_km['time']==t]; d = at_t['event'].sum(); c = len(at_t)-d
        if n > 0 and d > 0: surv *= (1 - d/n)
        t_out.append(t); s_out.append(surv); n -= (d+c)
    return np.array(t_out), np.array(s_out)

## 2. Kaplan-Meier by Treatment & Stage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for trt, color in [('Treatment_A',COLORS['primary']),('Treatment_B',COLORS['info']),('Control',COLORS['danger'])]:
    sub = df[df['Treatment']==trt]; t,s = kaplan_meier(sub['Time_Days'].values, sub['Event'].values)
    axes[0].step(t, s, where='post', linewidth=2.5, color=color, label=f'{trt} (n={len(sub)})')
axes[0].set_title('KM by Treatment', fontweight='bold'); axes[0].legend(); axes[0].set_ylim(0,1.05)
for stg, color in [('I',COLORS['primary']),('II',COLORS['info']),('III',COLORS['accent']),('IV',COLORS['danger'])]:
    sub = df[df['Stage']==stg]; t,s = kaplan_meier(sub['Time_Days'].values, sub['Event'].values)
    axes[1].step(t, s, where='post', linewidth=2.5, color=color, label=f'Stage {stg} (n={len(sub)})')
axes[1].set_title('KM by Stage', fontweight='bold'); axes[1].legend(); axes[1].set_ylim(0,1.05)
plt.tight_layout(); plt.show()

## 3. Hazard Ratios

In [ ]:
df_m = df.copy()
for c in ['Gender','Treatment','Stage','Smoking']:
    le = LabelEncoder(); df_m[c+'_e'] = le.fit_transform(df_m[c])
feat = ['Age','Gender_e','Treatment_e','Stage_e','BMI','Smoking_e','Comorbidities','BiomarkerA','BiomarkerB','PerformanceScore']
X = df_m[feat]; y = df_m['Event']
sc = StandardScaler(); X_sc = sc.fit_transform(X)
lr = LogisticRegression(max_iter=1000, random_state=42); lr.fit(X_sc, y)
hr = np.exp(lr.coef_[0])
fig, ax = plt.subplots(figsize=(10, 7))
hr_s = pd.Series(hr, index=feat).sort_values()
ax.barh(range(len(hr_s)), hr_s.values-1, left=1, color=[COLORS['danger'] if v>1 else COLORS['primary'] for v in hr_s.values], edgecolor='white')
ax.axvline(1, color=COLORS['dark'], linewidth=2, linestyle='--')
ax.set_yticks(range(len(hr_s))); ax.set_yticklabels(hr_s.index)
ax.set_title('Hazard Ratios', fontweight='bold'); ax.set_xlabel('HR')
for i,v in enumerate(hr_s.values): ax.text(v+0.02, i, f'{v:.2f}', va='center', fontweight='bold', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Predictive Modeling (2-Year Event)

In [ ]:
df_m['Event_2yr'] = ((df_m['Time_Days']<=730)&(df_m['Event']==1)).astype(int)
y2 = df_m['Event_2yr']
X_tr,X_te,y_tr,y_te = train_test_split(X,y2,test_size=0.2,random_state=42,stratify=y2)
sc2 = StandardScaler(); X_tr_s=sc2.fit_transform(X_tr); X_te_s=sc2.transform(X_te)
models = {'LogReg':LogisticRegression(max_iter=1000,random_state=42),'RF':RandomForestClassifier(n_estimators=200,max_depth=10,random_state=42,n_jobs=-1),'GB':GradientBoostingClassifier(n_estimators=150,max_depth=5,random_state=42)}
for n,m in models.items():
    if 'Log' in n: m.fit(X_tr_s,y_tr); yp=m.predict_proba(X_te_s)[:,1]
    else: m.fit(X_tr,y_tr); yp=m.predict_proba(X_te)[:,1]
    print(f"{n}: AUC={roc_auc_score(y_te,yp):.4f}, Brier={brier_score_loss(y_te,yp):.4f}")

## 5. Risk Stratification & Calibration

Model-based risk scores stratify patients into Low/Medium/High risk groups, validated by observed KM curves and event rates.

## 6. Conclusions

### Treatment A shows clear survival benefit over Control
### Stage IV has 3x+ hazard vs Stage I
### Smoking is a strong independent risk factor
### GB model achieves AUC ~0.73 for 2-year prediction
### Future: Cox PH with lifelines, competing risks, time-varying covariates

---
*Synthetic data for portfolio demonstration.*